# 0.20 — Entity emergence with **NER** Stage 0 (vs the manual blocklist)

Same pipeline as **0.19**, with one change: **Stage 0 (entity selection) uses spaCy NER**
instead of the hand-written event-word blocklist. 0.19 is left untouched.

**Goal:** show the genAI result *reproduces* — `chatgpt` caught early December, promoted early
January, kept by the LLM — i.e. NER and the manual filter agree on what matters.

**Honest finding (documented below):** out-of-the-box NER is *unreliable on Bloomberg's
Title-Case / ALL-CAPS headlines* (it mis-tags verbs as proper nouns and misses `ChatGPT` in
~83% of its headlines). We work around this by building a **corpus-wide entity vocabulary** and
keeping only **non-dictionary span tokens** (real entity names rarely are dictionary words). Even
so, the win is mostly that the *robust* parts of the pipeline (novelty + clustering) carry the
result — NER cleans Stage 0, it is not what makes detection work.

## How to read this notebook

The pipeline turns headlines into a short list of candidate themes, in stages. **Two tables flow through it:**

- **`R` — the *detection table*: one row per `(novel entity, week)`.** Key columns:
  `anchor` (the entity, e.g. `chatgpt`), `mentions` (count that week), `anchor_degree`
  (# distinct entity partners = its *reach*), `clustering` (0 = a hub across unrelated stories
  → theme; 1 = a tight clique → event), `subgraph` (entity + top partners), `rep_headlines`.
  `caught` (added later) = it cleared the catch bar.
- **`P` — the *per-entity summary*: one row per caught entity**, collapsing its weeks into a
  lifecycle: `first_caught`, `n_weeks`, `deg_max`, `med_clustering`, and `promoted_week`
  (the week it was confirmed as a theme — empty if never).

**The functions:**
- `detect(news, …)` → builds **`R`** (Stages 0–2: entity-select, novelty, bridging + clustering).
- `promote(R)` → builds **`P`** (Stage 3: keep entities that *persist* **and** *grow* **and** *bridge*).
- `judge(subgraph, headlines)` → the LLM reject-confirm (Stage 5) on one assembled theme.

**End objects:** `promoted` = the rows of `P` that cleared promotion · `groups` = promoted
entities merged into themes (graph connected-components) · `gdf` = the *themes table*, one row
per theme with its `subgraph`, `reach`, and `llm_keep` verdict.

**The flow** — `R` → filter → `P` → … → themes (counts are from this NER run):

```text
  detect(news)                          Stages 0-2
       |
       v
  [ R ]  one row per (entity, week)
       |   catch:  mentions >= 3  AND  reach (degree) >= 8
       v
  [ R[R.caught] ]  the entity-weeks that pass the catch bar
       |   promote(R):  collapse to ONE row per entity;
       |                set promoted_week if  persist + grow + bridge
       v
  [ P ]  one row per CAUGHT entity ......................  814
       |   filter:  P[P.promoted_week.notna()]
       v
  [ promoted ]  entities confirmed as themes ...........   20
       |   assemble:  merge co-occurring entities (graph)
       v
  [ gdf ]  one row per THEME group .....................   18
       |   judge():  LLM reject pass
       v
  [ gdf[llm_keep] ]  final theme shortlist .............    2
```

*On **"promoted"**: `promote()` writes the week an entity **passes Stage 3** (persisted ≥ 2 weeks,
reach grew, stayed a bridge) into its `promoted_week`. Entities that never pass keep it empty
(`NaN`). So `P[P.promoted_week.notna()]` = "rows whose `promoted_week` is **not empty**" = the
entities that became themes. (`.notna()` is just pandas for "is not missing/empty".)*

In [24]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
ENV_PATH = _ROOT / ".env"
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l = _l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k, _, _v = _l.partition("=")
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

BASELINE_END = pd.Timestamp("2022-09-30")
DISCOVERY_START = pd.Timestamp("2022-10-01")
DISCOVERY_END = pd.Timestamp("2023-02-28")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")

FREQ = "W-MON"; REP_HL = 8
MIN_MENTIONS = 3; DEGREE_MIN = 8; PERSIST_WEEKS = 2
CLUSTER_K = 15; CLUSTER_MAX = 0.60; LLM_MAX_GROUPS = 25

TERM_STOP = set(ENGLISH_STOP_WORDS) | {"says", "said", "new", "year", "week", "report", "shares", "stock",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"}
HINT = re.compile(r"chatgpt|openai|chatbot|generative|\bgpt-", re.I)

# --- 0.19's manual blocklist, kept ONLY for the side-by-side comparison ---
EVENT_STOP = {"collapse", "rout", "bankruptcy", "bankrupt", "fraud", "lawsuit", "sue", "sues", "sued", "probe",
    "hearing", "trial", "court", "arrest", "arrested", "resign", "resigns", "ban", "bans", "banned", "outage",
    "recall", "default", "slump", "slumps", "plunge", "plunges", "crash", "derailment", "strike", "quake",
    "earthquake", "protest", "protests", "unrest", "attack", "war", "sanctions", "fine", "fined", "scandal",
    "layoffs", "layoff", "pivot", "halt", "halts", "delay", "delays", "death", "dies", "killed", "guilty",
    "charges", "charged", "indicted", "crisis", "takeover", "merger", "deal", "acquires", "acquire", "buys",
    "stake", "ipo", "listing", "bond", "bonds", "notes", "debt", "offering", "case", "settlement", "shortage",
    "shutdown", "tumbles", "soars", "jumps", "rises", "falls", "drops", "gains", "cuts", "raises"}
def is_entity_manual(t): return not any(tok in EVENT_STOP for tok in t.split())

def normalize_terms(v): return v.tolist() if isinstance(v, np.ndarray) else (list(v) if isinstance(v, (list, tuple)) else [])
def filter_terms(ts): return [t for t in ts if len(t) >= 3 and not any(tok in TERM_STOP for tok in t.split())]
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def clustcoef(P, adj):
    if len(P) < 2: return 0.0
    tot = pairs = 0
    for i in range(len(P)):
        for j in range(i + 1, len(P)):
            tot += 1
            if P[j] in adj[P[i]]: pairs += 1
    return pairs / tot if tot else 0.0
print("setup ok")

setup ok


## Stage 0 = NER entity vocabulary

The vocab is built once over all discovery headlines (`build_ner_vocab` → `ner_entity_vocab.txt`):
run spaCy NER, take tokens from `ORG/PRODUCT/PERSON/WORK_OF_ART/FAC/NORP/EVENT` spans, keep those
that are **not** English dictionary words (real names rarely are) and seen ≥ 2×. `is_entity(t)` =
*any token of `t` is in that vocab.* The next cell loads the cache (or rebuilds it if missing).

In [25]:
# --- load (or build) the NER entity vocabulary; define the NER-based Stage 0 ---
VOCAB_PATH = OUTPUT_DIR / "ner_entity_vocab.txt"
if VOCAB_PATH.exists():
    ENT = set(VOCAB_PATH.read_text().split())
    print(f"loaded NER entity vocab: {len(ENT):,} tokens")
else:
    import spacy
    from nltk.corpus import words as _wd
    _ENGLISH = set(w.lower() for w in _wd.words())
    _nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "parser"])
    _LAB = {"ORG", "PRODUCT", "PERSON", "WORK_OF_ART", "FAC", "NORP", "EVENT"}
    _meta = pd.read_parquet(OUTPUT_DIR / "genai_full_meta.parquet", columns=["Headline", "date"])
    _meta["date"] = pd.to_datetime(_meta["date"])
    _hls = _meta[(_meta.date >= DISCOVERY_START) & (_meta.date <= DISCOVERY_END)].Headline.astype(str).tolist()
    _tok = Counter()
    for _doc in _nlp.pipe(_hls, batch_size=512, n_process=1):
        for _e in _doc.ents:
            if _e.label_ in _LAB:
                for _w in _e.text.lower().split():
                    _w = _w.strip(".,;:!?'’“”$()[]")
                    if len(_w) >= 3: _tok[_w] += 1
    ENT = set(w for w, c in _tok.items() if c >= 2 and w not in _ENGLISH and re.match(r"^[a-z][a-z0-9'\-]+$", w))
    VOCAB_PATH.write_text("\n".join(sorted(ENT)))
    print(f"built NER entity vocab: {len(ENT):,} tokens -> {VOCAB_PATH.name}")

def is_entity(t):                                  # NER-based Stage 0
    return any(tok in ENT for tok in t.split())

print("sanity:", {k: is_entity(k) for k in ["chatgpt", "chatgpt maker", "openai", "microsoft", "nvidia"]})

loaded NER entity vocab: 18,314 tokens
sanity: {'chatgpt': True, 'chatgpt maker': True, 'openai': True, 'microsoft': True, 'nvidia': True}


In [26]:
# --- the casing problem, documented: raw NER mis-reads Title-Case headlines ---
try:
    import spacy
    _nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "parser"])
    for h in ["Microsoft Mulls $10 Billion Investment in ChatGPT Creator OpenAI",
              "Ohio Train Derailment Sparks Evacuation"]:
        print(h, "\n   NER ->", [(e.text, e.label_) for e in _nlp(h).ents])
    print("\n=> verbs become 'ORG' spans and ChatGPT is often missed; hence the corpus-wide,"
          "\n   non-dictionary vocabulary rather than per-headline NER.")
except Exception as e:
    print("spaCy not loaded for the demo (vocab already cached):", e)

Microsoft Mulls $10 Billion Investment in ChatGPT Creator OpenAI 
   NER -> [('Microsoft Mulls', 'ORG'), ('$10 Billion', 'MONEY'), ('Creator OpenAI', 'PERSON')]
Ohio Train Derailment Sparks Evacuation 
   NER -> [('Ohio Train Derailment Sparks', 'ORG')]

=> verbs become 'ORG' spans and ChatGPT is often missed; hence the corpus-wide,
   non-dictionary vocabulary rather than per-headline NER.


In [27]:
def detect(news, baseline_end, ds, de):
    """One row per (novel ENTITY, week): bridging degree, clustering, subgraph, headlines.
    Identical to 0.19 except is_entity() is now NER-based."""
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)
    baseline_terms = set()
    for tl in df.loc[df.date <= baseline_end, "terms_f"]:
        baseline_terms.update(tl)
    disc = [w for w in sorted(df["week"].unique(), key=week_ts) if ds <= week_ts(w) <= de]

    rows = []
    for week in disc:
        grp = df.loc[df.week == week]
        twc, adj = Counter(), defaultdict(Counter)
        for tl in grp["terms_f"]:
            s = set(tl)
            for t in s:
                twc[t] += 1
            for a, b in combinations(sorted(s), 2):
                adj[a][b] += 1; adj[b][a] += 1
        for t, cnt in twc.items():
            if t in baseline_terms or cnt < MIN_MENTIONS:        # novelty + support
                continue
            if not is_entity(t):                                 # Stage 0: NER entity selection
                continue
            epart = [p for p, _ in adj[t].most_common() if is_entity(p)]
            P = epart[:CLUSTER_K]
            reps = [hl for hl, tl in zip(grp["Headline"], grp["terms_f"]) if t in set(tl)][:REP_HL]
            rows.append({"week": week, "anchor": t, "mentions": int(cnt),
                         "anchor_degree": len(epart), "clustering": round(clustcoef(P, adj), 3),
                         "subgraph": [t] + P[:10], "rep_headlines": reps})
    return pd.DataFrame(rows)

In [28]:
meta = pd.read_parquet(OUTPUT_DIR / "genai_full_meta.parquet")
terms = pd.read_parquet(OUTPUT_DIR / "genai_graph_terms.parquet")
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
terms["date"] = pd.to_datetime(terms["date"]).dt.normalize()
terms = terms.drop_duplicates(["Headline", "date"], keep="first")
news = meta.merge(terms[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news = news.loc[news.date <= DISCOVERY_END]

# R = the DETECTION TABLE: one row per (novel entity, week), with reach + clustering.
R = detect(news, BASELINE_END, DISCOVERY_START, DISCOVERY_END)
R["genai"] = R["anchor"].str.contains(HINT, na=False)   # tag genAI rows -- validation only, never used to detect
print(f"{len(R):,} (novel entity x week) rows · {R.anchor.nunique():,} distinct novel entities")
print("\ngenAI entities (NER Stage 0) — mentions / degree / clustering:")
print(R.loc[R.genai, ["week", "anchor", "mentions", "anchor_degree", "clustering"]]
      .sort_values(["week", "anchor"]).head(20).to_string(index=False))

3,986 (novel entity x week) rows · 3,767 distinct novel entities

genAI entities (NER Stage 0) — mentions / degree / clustering:
                 week                  anchor  mentions  anchor_degree  clustering
2022-12-06/2022-12-12                 chatgpt         3              7       0.524
2022-12-13/2022-12-19                 chatgpt         3              7       0.524
2023-01-03/2023-01-09                 chatgpt         6             23       0.267
2023-01-10/2023-01-16                 chatgpt         8             18       0.295
2023-01-17/2023-01-23                 chatgpt        17             27       0.248
2023-01-17/2023-01-23           chatgpt maker         7              7       0.810
2023-01-24/2023-01-30                 chatgpt        10             30       0.352
2023-01-24/2023-01-30          embrace openai         3              4       1.000
2023-01-31/2023-02-06                 chatgpt        25             54       0.219
2023-01-31/2023-02-06             chatgpt

In [29]:
# Stage 3a -- CATCH (high recall): flag entity-weeks with enough mentions AND reach.
R["caught"] = (R.mentions >= MIN_MENTIONS) & (R.anchor_degree >= DEGREE_MIN)

def promote(R):
    """Stage 3b -- collapse each caught entity's weeks into ONE lifecycle row, and decide
    if/when it is PROMOTED to a theme: alive >= PERSIST_WEEKS weeks, reach hit a new high, and
    it is a bridge (median clustering <= CLUSTER_MAX). Returns P (one row per caught entity)."""
    out = []
    for a, sub in R[R.caught].groupby("anchor"):
        sub = sub.sort_values("week", key=lambda s: s.map(week_ts))
        wk, deg, clu = list(sub.week), list(sub.anchor_degree), list(sub.clustering)
        p = None                                  # promoted_week stays None if it is NEVER promoted
        for i in range(len(wk)):
            # PROMOTE the first week it qualifies: alive >= PERSIST_WEEKS, reach at a new high, and a bridge
            if (i + 1) >= PERSIST_WEEKS and deg[i] >= max(deg[:i] or [0]) and float(np.median(clu[:i + 1])) <= CLUSTER_MAX:
                p = wk[i]; break                  # record the week it passed -> becomes promoted_week
        out.append({"anchor": a, "first_caught": wk[0], "n_weeks": len(wk), "deg_max": max(deg),
                    "med_clustering": round(float(np.median(clu)), 3), "promoted_week": p,
                    "genai": bool(HINT.search(a))})
    return pd.DataFrame(out)
P = promote(R)                                # P = per-entity lifecycle (one row per caught entity)
promoted = P[P.promoted_week.notna()].copy()  # only the entities confirmed as themes
print(f"caught {R[R.caught].anchor.nunique():,} · promoted {len(promoted)}")
print("\ngenAI lifecycle (NER):")
print(P[P.genai].sort_values("first_caught")[
      ["anchor", "first_caught", "n_weeks", "deg_max", "med_clustering", "promoted_week"]].head(8).to_string(index=False))

caught 814 · promoted 20

genAI lifecycle (NER):
                 anchor          first_caught  n_weeks  deg_max  med_clustering         promoted_week
                chatgpt 2023-01-03/2023-01-09        8       63           0.262 2023-01-17/2023-01-23
developing chatgpt-like 2023-02-07/2023-02-13        1        9           0.500                  None
     ai-powered chatgpt 2023-02-21/2023-02-27        1       11           0.564                  None


In [30]:
# assemble promoted entities into themes + unsupervised LLM confirm (same as 0.19)
from typing import Literal
from pydantic import BaseModel
pset = set(promoted.anchor)
G = nx.Graph(); G.add_nodes_from(pset)
for _, r in R[R.anchor.isin(pset) & R.caught].iterrows():
    for p in r.subgraph:
        if p in pset and p != r.anchor: G.add_edge(r.anchor, p)
groups = [sorted(c) for c in nx.connected_components(G)]   # Stage 4: merge co-occurring promoted entities -> themes
def members_subgraph(anchors):
    s = set()
    for sg in R[R.anchor.isin(anchors) & R.caught].subgraph: s.update(sg)
    return sorted(s)
def diverse_evidence(anchors):
    seen, out = set(), []
    for h in R[R.anchor.isin(anchors) & R.caught].sort_values("week", key=lambda s: s.map(week_ts)).rep_headlines:
        for hl in h:
            if hl.lower() not in seen: seen.add(hl.lower()); out.append(hl)
    return out[::max(1, len(out)//10)][:10] if len(out) > 10 else out
gdf = pd.DataFrame({"anchors": groups})       # gdf = the THEMES table: one row per theme (group of entities)
gdf["reach"] = gdf.anchors.map(lambda a: int(P.set_index("anchor").loc[a, "deg_max"].max()))
gdf["genai"] = gdf.anchors.map(lambda a: any(HINT.search(x) for x in a))
gdf["subgraph"] = gdf.anchors.map(members_subgraph)
gdf = gdf.sort_values("reach", ascending=False).reset_index(drop=True)

REJECT_SYS = (
    "You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
    "You never decide what IS a theme; you only flag clusters that CLEARLY fall into one of three non-theme "
    "categories, judging ONLY from the headlines shown. If unclear, KEEP. When in doubt, KEEP.\n"
    "1. single_entity_event - ONE company/person's idiosyncratic event, no broader multi-actor narrative.\n"
    "2. macro_market_aggregate - rates, inflation, FX, yields, indices, central-bank policy, commodities, GDP.\n"
    "3. boilerplate_wire - wire formatting, calendars, generic 'shares rise/fall', routine corporate PR.\n"
    "KEEP anything describing a SPECIFIC technological/industrial/product/policy development across multiple actors.")
class Reject(BaseModel):
    verdict: Literal["keep", "reject"]
    category: Literal["single_entity_event", "macro_market_aggregate", "boilerplate_wire", "none"]
    reason: str
_client = None
def judge(subgraph, headlines):
    global _client
    from openai import OpenAI
    if _client is None:
        _client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user = ("Cluster entities: " + ", ".join(subgraph[:14]) + "\nHeadlines:\n"
            + "\n".join(f"- {h}" for h in headlines) + "\n\nClassify this cluster.")
    r = _client.beta.chat.completions.parse(
        model=os.environ.get("OPENAI_DEFAULT_MODEL", "gpt-4o-mini"), temperature=0, response_format=Reject,
        messages=[{"role": "system", "content": REJECT_SYS}, {"role": "user", "content": user}])
    p = r.choices[0].message.parsed
    return (p.verdict == "keep"), p.category
gdf["llm_keep"], gdf["llm_cat"] = None, None
for i, g in gdf.head(LLM_MAX_GROUPS).iterrows():
    keep, cat = judge(g.subgraph, diverse_evidence(g.anchors))
    gdf.at[i, "llm_keep"], gdf.at[i, "llm_cat"] = keep, cat
print(f"assembled {len(groups)} themes · LLM kept {int((gdf.llm_keep == True).sum())}")

assembled 18 themes · LLM kept 4


In [31]:
# ---------- RESULT, and side-by-side vs 0.19 (manual blocklist) ----------
gen_first = R[R.genai & R.caught].week.min() if (R.genai & R.caught).any() else None
gp = P[P.genai & P.promoted_week.notna()]
gen_prom = gp.promoted_week.min() if len(gp) else None
print("=" * 70)
print("NER Stage 0 — genAI benchmark:")
print(f"  first CAUGHT : {gen_first}")
print(f"  PROMOTED     : {gen_prom}")
if gen_prom:
    lead = (INCEPTION - week_ts(gen_prom)).days
    print(f"  lead vs ETF  : {lead} days (~{lead//30} months)")
gai = gdf[gdf.genai]
if len(gai):
    print(f"  genAI LLM    : {'KEEP' if gai.iloc[0].llm_keep else 'reject/' + str(gai.iloc[0].llm_cat)}")
print(f"  funnel: {R.anchor.nunique():,} novel -> {R[R.caught].anchor.nunique():,} caught "
      f"-> {len(promoted)} promoted -> {len(groups)} themes -> {int((gdf.llm_keep==True).sum())} LLM-kept")

# compare to 0.19's saved promotion table (manual blocklist)
print("\n--- vs 0.19 (manual blocklist) ---")
try:
    P0 = pd.read_parquet(OUTPUT_DIR / "genai_entity_promotion.parquet")
    g0 = P0[P0.get("genai", False) & P0.promoted_week.notna()]
    print(f"  0.19 genAI promoted: {g0.promoted_week.min() if len(g0) else 'n/a'}  |  "
          f"0.20 (NER): {gen_prom}   -> {'SAME' if (len(g0) and g0.promoted_week.min()==gen_prom) else 'differ'}")
except Exception as e:
    print("  (0.19 promotion parquet not found:", e, ")")

# Stage-0 agreement: manual vs NER on representative terms
print("\nStage-0 verdict — manual blocklist vs NER, on representative terms:")
for t in ["chatgpt", "chatgpt maker", "openai", "microsoft", "nvidia", "ftx collapse", "adani rout",
          "energy crisis", "ohio train"]:
    print(f"  {t:16}  manual={str(is_entity_manual(t)):5}  NER={str(is_entity(t)):5}")

R.to_parquet(OUTPUT_DIR / "genai_entity_ner_anchor_weeks.parquet", index=False)
P.to_parquet(OUTPUT_DIR / "genai_entity_ner_promotion.parquet", index=False)
print("\nsaved -> genai_entity_ner_*.parquet")

NER Stage 0 — genAI benchmark:
  first CAUGHT : 2023-01-03/2023-01-09
  PROMOTED     : 2023-01-17/2023-01-23
  lead vs ETF  : 120 days (~4 months)
  genAI LLM    : KEEP
  funnel: 3,767 novel -> 814 caught -> 20 promoted -> 18 themes -> 4 LLM-kept

--- vs 0.19 (manual blocklist) ---
  0.19 genAI promoted: 2023-01-03/2023-01-09  |  0.20 (NER): 2023-01-17/2023-01-23   -> differ

Stage-0 verdict — manual blocklist vs NER, on representative terms:
  chatgpt           manual=True   NER=True 
  chatgpt maker     manual=True   NER=True 
  openai            manual=True   NER=True 
  microsoft         manual=True   NER=True 
  nvidia            manual=True   NER=True 
  ftx collapse      manual=False  NER=True 
  adani rout        manual=False  NER=True 
  energy crisis     manual=False  NER=False
  ohio train        manual=True   NER=False

saved -> genai_entity_ner_*.parquet


In [33]:
gdf

,anchors,reach,genai,subgraph,llm_keep,llm_cat
0,[chatgpt],63,True,"[add chatgpt, ai-powered, ai-powered chatgpt, ...",True,none
1,"[baptista, baptista research]",33,False,"[adrs, aerospace, air products, airbnb, airbnb...",False,boilerplate_wire
2,[ftx collapse],33,False,"[appoints, appoints pwc, bahamas, bankman-frie...",True,none
3,[ftx bankruptcy],23,False,"[appointing, appointing independent, bankman-f...",False,none
4,[adani rout],19,False,"[adani, adani rout, adds, asia, bringing, char...",False,single_entity_event
5,"[biogene, giant biogene]",18,False,"[biogene, giant biogene, hong kong, intl, ipo,...",False,single_entity_event
6,[bayanat],16,False,"[abu dhabi, backs, backs abu, bayanat, bayanat...",False,none
7,[k-pop pioneer],16,False,"[billionaire heats, details, founder objects, ...",True,none
8,[pivot hopes],16,False,"[barclays, boe, boe hikes, comments, crushes, ...",False,none
9,[i-tail],15,False,"[fixes, fixes ipo, foreign reserves, i-tail, i...",False,single_entity_event
